[![在 Colab 中打开](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Arize-ai/phoenix/blob/main/tutorials/tracing/smolagents_tracing_tutorial.ipynb)


<center>
    <p style="text-align:center">
        <img alt="Phoenix 标志" src="https://raw.githubusercontent.com/Arize-ai/phoenix-assets/9e6101d95936f4bd4d390efc9ce646dc6937fb2d/images/socal/github-large-banner-phoenix.jpg" width="1000"/>
        <br>
        <br>
        <a href="https://arize.com/docs/phoenix/">文档</a>
        |
        <a href="https://github.com/Arize-ai/phoenix">GitHub</a>
        |
        <a href="https://join.slack.com/t/arize-ai/shared_invite/zt-3r07iavnk-ammtATWSlF0pSrd1DsMW7g">社区</a>
    </p>
</center>
<h1 align="center">使用 Arize Phoenix 追踪 Smolagents</h1>

## 安装依赖

In [ ]:
!pip install smolagents -q
!pip install -q arize-phoenix opentelemetry-sdk opentelemetry-exporter-otlp openinference-instrumentation-smolagents

## 连接 Phoenix


本示例使用 Phoenix 作为追踪数据的目的地。采用这种方式时，也可以添加其他需要的导出器。

如需为 Phoenix 设置 API 密钥，请前往[此处](https://app.phoenix.arize.com/)。

#### 密钥

注意：运行此 Colab 需要：

* Phoenix API 密钥
* Hugging Face Token

In [ ]:
import getpass
import os

# 尚未设置 API 密钥时提示用户输入
if not (phoenix_api_key := os.getenv("PHOENIX_API_KEY")):
    phoenix_api_key = getpass.getpass("请输入 Phoenix API 密钥： ")
if not (hf_token_value := os.getenv("HF_TOKEN")):
    hf_token_value = getpass.getpass("请输入 Hugging Face 令牌： ")

# 使用提供的密钥设置环境变量
os.environ["PHOENIX_CLIENT_HEADERS"] = f"api_key={phoenix_api_key}"
os.environ["PHOENIX_COLLECTOR_ENDPOINT"] = "https://app.phoenix.arize.com"
os.environ["HF_TOKEN"] = f"{hf_token_value}"

### 定义追踪提供程序

In [ ]:
from phoenix.otel import register

tracer_provider = register(
    project_name="my-agent-app",  # 默认值为 'default'
)

## 为 Smolagents 添加埋点



In [ ]:
from openinference.instrumentation.smolagents import SmolagentsInstrumentor

SmolagentsInstrumentor().instrument(tracer_provider=tracer_provider)

## 创建并运行 Smolagents


现在可以使用 smolagents 模块创建各种智能体，Phoenix 会追踪智能体的每一步、输入、输出、工具及调用。

In [ ]:
from smolagents import (
    CodeAgent,
    InferenceClientModel,
    ToolCallingAgent,
    VisitWebpageTool,
    WebSearchTool,
)

model = InferenceClientModel()

search_agent = ToolCallingAgent(
    tools=[WebSearchTool(), VisitWebpageTool()],
    model=model,
    name="search_agent",
    description="这是一个可以进行网络搜索的智能体。",
)
manager_agent = CodeAgent(
    tools=[],
    model=model,
    managed_agents=[search_agent],
)
manager_agent.run("Based on the latest news, what is happening in extraterrestrial life?")

 ## ⭐⭐⭐ 如果本指南对你有帮助，请为 [Arize Phoenix](https://github.com/Arize-ai/phoenix) 点亮一颗星 ⭐⭐⭐